# Día 19 — Práctica: Limpieza y Joins en pandas

**Dataset:** Empresa de servicios — `empleados.csv` + `proyectos.csv`  
**Instrucciones:** resuelve cada ejercicio sin ver el notebook de referencia. El resultado esperado está descrito en cada celda.

---

In [1]:
import pandas as pd

dfEmpleados = pd.read_csv('../data/day19/empleados.csv')
dfProyectos  = pd.read_csv('../data/day19/proyectos.csv')

print('Empleados:', dfEmpleados.shape)
print('Proyectos:', dfProyectos.shape)
print('--------------------------------')
print('Empleados:', dfEmpleados.dtypes)
print('________________________________')
print('Proyectos:', dfProyectos.dtypes)
print('--------------------------------')
print('Información estructurada sobre el dataset: ')
print('--------------------------------')
print('Empleados:', dfEmpleados.info())
print('________________________________')
print('Proyectos:', dfProyectos.info())
print('--------------------------------')
print('Primeros registros y nombre de cada campo: ')
print('--------------------------------')
print('Empleados:', dfEmpleados.head())
print('________________________________')
print('Proyectos:', dfProyectos.head())
print('--------------------------------')


Empleados: (50, 5)
Proyectos: (50, 6)
--------------------------------
Empleados: empleado_id      int64
nombre             str
departamento       str
salario            str
fecha_ingreso      str
dtype: object
________________________________
Proyectos: proyecto_id         int64
empleado_id         int64
cliente               str
presupuesto       float64
estado                str
duracion_meses    float64
dtype: object
--------------------------------
Información estructurada sobre el dataset: 
--------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   empleado_id    50 non-null     int64
 1   nombre         50 non-null     str  
 2   departamento   44 non-null     str  
 3   salario        45 non-null     str  
 4   fecha_ingreso  42 non-null     str  
dtypes: int64(1), str(4)
memory usage: 2.1 KB
Empleados: None
__________________

---
## Ejercicio 1 — Detectar nulos

Calcula cuántos nulos hay por columna en `empleados` y qué porcentaje representa cada uno.

**Resultado esperado:** tabla con columnas `nulos` y `porcentaje_%`, mostrando solo las filas donde hay al menos un nulo.

In [2]:
# La identificación de nulos se con isnull(), que devuelve True donde hay NaN, y luego .sum() cuenta esos True
nulos = dfEmpleados.isnull().sum()

#Para saber el porcentaje de nulos se dicide la cantidad de nulos acumulada por el total de registros
ptcNulos = (nulos/ len(dfEmpleados)*100).round(2) 

#Se crea una tabla de muestreo con campos filtrados/nuevos 
# con un diccionario, donde cada clave equivale a una columna
resumen = pd.DataFrame({'nulos': nulos, 'porcentaje_%': ptcNulos})

# Se muestra el resumen filtrando unicamente por donde existen dentro de resumen
print(resumen[resumen['nulos']>0])

               nulos  porcentaje_%
departamento       6          12.0
salario            5          10.0
fecha_ingreso      8          16.0


---
## Ejercicio 2 — Tratar nulos

En `empleados`:
- Los nulos en `departamento` → rellenar con `'Sin asignar'`
- Los nulos en `salario` → eliminar la fila
- Los nulos en `fecha_ingreso` → eliminar la fila

**Resultado esperado:** imprimir cuántas filas se eliminaron y verificar que no quedan nulos en esas columnas.

In [3]:
# Departamento nulo equivale a rellenar campo nulo con valor asignado por fillna() = 'Sin asignar'
dfEmpleados['departamento'] = dfEmpleados['departamento'].fillna('Sin asignar')

filas_antes = len(dfEmpleados)

dfEmpleados = dfEmpleados.dropna(subset=['salario'])
dfEmpleados = dfEmpleados.dropna(subset=['fecha_ingreso'])

print('Filas eliminadas por departamento nulo:', filas_antes - len(dfEmpleados))
print('Nulos restantes en salario, departamento y fecha de ingreso :')
print(dfEmpleados[['salario', 'departamento', 'fecha_ingreso']].isnull().sum())


Filas eliminadas por departamento nulo: 13
Nulos restantes en salario, departamento y fecha de ingreso :
salario          0
departamento     0
fecha_ingreso    0
dtype: int64


---
## Ejercicio 3 — Eliminar duplicados

`empleados.csv` contiene filas duplicadas. Identifícalas, muestra cuáles son y elimínalas.

**Resultado esperado:** número de duplicados encontrados, detalle de los empleado_id duplicados, shape final tras eliminarlos.

In [4]:
print('Filas duplicadas: ',dfEmpleados.duplicated().sum())
print("Duplicados: ")
print(dfEmpleados[dfEmpleados.duplicated(keep=False)][['empleado_id', 'nombre']].sort_values('empleado_id'))
dfEmpleados = dfEmpleados.drop_duplicates()
print("Estructura o Shape final: ", dfEmpleados.shape)


Filas duplicadas:  3
Duplicados: 
    empleado_id          nombre
0           201    Ana Martínez
47          201    Ana Martínez
1           202  Carlos Sánchez
48          202  Carlos Sánchez
2           203    Laura García
49          203    Laura García
Estructura o Shape final:  (34, 5)


---
## Ejercicio 4 — Corregir tipos

Verifica los tipos de `empleados` con `dtypes`. Hay dos columnas con tipo incorrecto:
- `salario` está como texto (tiene el símbolo `€`) → convertir a `float`
- `fecha_ingreso` está como texto (formato `dd-mm-yyyy`) → convertir a `datetime`

**Resultado esperado:** mostrar `dtypes` antes y después de corregir.

In [5]:
print("Tipos actuales en 'Empleados': ")
dfEmpleados.dtypes

dfEmpleados['salario'] = dfEmpleados['salario'].replace('€', '', regex=False).astype(float)
dfEmpleados['fecha_ingreso'] = pd.to_datetime(dfEmpleados['fecha_ingreso'], format='%d-%m-%Y')


print('Tipos corregidos:')
print(dfEmpleados[['fecha_ingreso', 'salario']].dtypes)
print(dfEmpleados[['fecha_ingreso', 'salario']].head(5))





Tipos actuales en 'Empleados': 


ValueError: could not convert string to float: '42000€'

---
## Ejercicio 5 — INNER merge

Se une `proyectos` con `empleados` usando la columna común `empleado_id`.


In [ ]:
detalles_proyectos = pd.merge(dfEmpleados, dfProyectos, on='empleado_id', how='inner')
print('Empleados originales: ', len(dfEmpleados))
print('Resultado del merge: ', len(detalles_proyectos))
print('Primeras filas:')

print(detalles_proyectos[['proyecto_id', 'empleado_id', 'nombre', 'departamento', 'presupuesto']].head())

Empleados originales:  34
Resultado del merge:  34
Primeras filas:
   proyecto_id  empleado_id          nombre departamento  presupuesto
0            1          201    Ana Martínez       Ventas      15000.0
1           17          202  Carlos Sánchez  Sin asignar          NaN
2           33          203    Laura García    Marketing       7500.0
3            2          204   Miguel Torres           IT      32000.0
4           18          205     Isabel Ruiz         RRHH      19000.0


---
## Ejercicio 6 — Comparar inner vs left merge

Se realiza los dos tipos de merge entre `proyectos` y `empleados`. Finalmente se compara cuántas filas devuelve cada uno e identifica qué proyectos quedan fuera del inner merge.


In [9]:
detalles_proyectos = pd.merge(dfEmpleados, dfProyectos, on='empleado_id', how='left')
print('Empleados originales: ', len(dfEmpleados))
print('Resultado del merge: ', len(detalles_proyectos))
print('Primeras filas:')

print(detalles_proyectos[['proyecto_id', 'empleado_id', 'nombre', 'departamento', 'presupuesto']].head())

Empleados originales:  34
Resultado del merge:  34
Primeras filas:
   proyecto_id  empleado_id          nombre departamento  presupuesto
0            1          201    Ana Martínez       Ventas      15000.0
1           17          202  Carlos Sánchez  Sin asignar          NaN
2           33          203    Laura García    Marketing       7500.0
3            2          204   Miguel Torres           IT      32000.0
4           18          205     Isabel Ruiz         RRHH      19000.0
